# Full 20 Newsgroups — large-corpus cumulative supervised learning with TeAL

**Local, long-running acceptance notebook — deliberately excluded from GitHub's short notebook CI.**

This notebook uses **all 20 categories** of the real 20 Newsgroups dataset: approximately **11,314 development documents** and **7,532 independent test documents** (18,846 total). It executes a complete TeAL workflow:

1. Import the real training text and labels into a TeAL Project.
2. Fit text representations **only on training documents**: sparse count features, sparse TF-IDF, dense SVD, dense LDA topic mixtures, and an augmented SVD feature matrix.
3. Search supervised classifiers with the **same development-only cross-validation folds** across representations.
4. Finalize the selected estimator on every development label.
5. Only after finalization, load the separate 20 Newsgroups **test** partition and transform it using the *frozen training operators*.
6. Predict on the held-out test texts, join predictions to gold labels by stable TeAL keys, inspect per-class outcomes, and reopen the frozen model.

**Methodological distinction:** This is a genuine independent text-level test set: no test text or labels enter any learned representation or model selection. Within development CV, vocabulary/TF-IDF/SVD/LDA are fitted once on all **unlabeled development documents**, not refitted inside each fold. Thus CV evaluates fixed development representations rather than nested preprocessing pipelines. The separately held-out test set is the primary external check.

Scikit-learn downloads and caches 20 Newsgroups automatically the first time it is requested. A network connection is needed only for that first fetch. The dataset contains historical Usenet material that may include offensive text. We apply `remove=('headers', 'footers', 'quotes')` to reduce header-based label leakage, though this heuristic is imperfect. Source: [scikit-learn 20 Newsgroups documentation](https://scikit-learn.org/stable/datasets/real_world.html#the-20-newsgroups-text-dataset).

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import tempfile
import json

import numpy as np
import pandas as pd
import sklearn
from packaging.version import Version
from scipy import sparse
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

from text_analysis_lab import Project
from text_analysis_lab.analysis import ClassifierSearch
from text_analysis_lab.translators import (
    CountVectorizer, TfidfTransformer, SVD, LDA, FeatureMatrix,
)

# Deliberately full dataset by default. Set limits only when debugging.
MAX_DEVELOPMENT = None
MAX_TEST = None
MAX_VOCAB = 15_000
SVD_COMPONENTS = 64
LDA_TOPICS = 20
LDA_ITERATIONS = 8
CV_FOLDS = 3
QUERY_BATCH_SIZE = 512

# L1 and Elastic Net can take longer to converge than the original L2 baseline.
REGULARIZATION_C = [0.1, 1.0]
LOGISTIC_L1_RATIOS = [0.0, 0.5, 1.0]  # L2, Elastic Net, L1
LOGISTIC_MAX_ITER = 2500
LOGISTIC_TOL = 1e-3
RANDOM_STATE = 2025

# LDA is the slowest stage. Set False to run counts/TF-IDF/SVD/augmented only.
INCLUDE_LDA = True

WORK = Path(tempfile.mkdtemp(prefix="teal-20newsgroups-full-"))
PROJECT_PATH = WORK / "teal_project"
timings = {}
print("Working directory:", WORK)
print("Training/test limits:", MAX_DEVELOPMENT, MAX_TEST)
print("LDA enabled:", INCLUDE_LDA)
print("scikit-learn version:", sklearn.__version__)

## 1. Fetch the *development* partition and inspect class balance

The training and test subsets are predefined by scikit-learn. We deliberately do **not** load the test subset yet. We strip identifying Usenet headers, footers, and quoted replies to reduce superficial topic clues.

The TeAL table contains integer `label` as metadata, never as a feature. `log_words` is a simple text-derived numerical feature.

In [ ]:
started = perf_counter()
train_news = fetch_20newsgroups(
    subset="train", categories=None, shuffle=True, random_state=RANDOM_STATE,
    remove=("headers", "footers", "quotes"),
)
train_texts = list(train_news.data)
train_targets = np.asarray(train_news.target, dtype=int)
if MAX_DEVELOPMENT is not None:
    train_texts = train_texts[:MAX_DEVELOPMENT]
    train_targets = train_targets[:MAX_DEVELOPMENT]
names = list(train_news.target_names)
assert len(names) == 20
print("Development observations:", len(train_texts), "categories:", len(names))
display(pd.Series(train_targets).map(lambda i: names[int(i)]).value_counts().to_frame("n"))
timings["load_train"] = perf_counter() - started

In [ ]:
def make_input_frame(texts, *, labels=None):
    cleaned = [text if text.strip() else " " for text in texts]
    lengths = np.array([len(t.split()) for t in cleaned], dtype=int)
    frame = pd.DataFrame({
        "text": cleaned,
        "record_id": np.arange(len(cleaned), dtype=int),
        "log_words": np.log1p(lengths).astype(float),
    })
    if labels is not None:
        frame["label"] = np.asarray(labels, dtype=int)
    return frame

train_df = make_input_frame(train_texts, labels=train_targets)
train_csv = WORK / "development.csv"
train_df.to_csv(train_csv, index=False)
project = Project.create(PROJECT_PATH, name="full_twenty_newsgroups")
development = project.read_csv(
    train_csv, text_fields="text",
    metadata_fields=["record_id", "log_words", "label"],
    batch_size=QUERY_BATCH_SIZE,
)
assert development.n_rows == len(train_df)
print("Development artifact:", development.artifact_id)
print("Development keys:", development.primary_key)

## 2. Construct real text representations on development documents

The order here matters: fit CountVectorizer, then TF-IDF on the counts, then SVD on TF-IDF and optionally LDA on counts. These operators are frozen and can later transform new documents using identical vocabulary and feature axes.

SVD and LDA are **different** families: SVD provides signed orthogonal directions in weighted lexical space; LDA provides nonnegative document-topic mixtures. `FeatureMatrix` appends the log word count to SVD coordinates without duplicating the dense numerical matrix.

In [ ]:
started = perf_counter()
vectorizer = CountVectorizer(
    text_field="text", lowercase=True, stop_words="english",
    min_df=2, max_features=MAX_VOCAB,
)
counts_train = project.translate(
    vectorizer, development, batch_size=QUERY_BATCH_SIZE,
)["output"]
timings["count_fit"] = perf_counter() - started
print("Counts:", counts_train.n_rows, "×", len(counts_train.get_data_columns()))
assert sparse.issparse(counts_train.get_matrix(positions=[0, 1]))

In [ ]:
started = perf_counter()
tfidf_operator = TfidfTransformer(norm="l2")
tfidf_train = project.translate(
    tfidf_operator, counts_train, batch_size=QUERY_BATCH_SIZE,
)["output"]
timings["tfidf_fit"] = perf_counter() - started

started = perf_counter()
svd_operator = SVD(n_components=SVD_COMPONENTS, random_state=RANDOM_STATE)
svd_train = project.translate(svd_operator, tfidf_train)["output"]
timings["svd_fit"] = perf_counter() - started
print("TF-IDF:", tfidf_train.get_matrix(positions=[0]).shape)
print("SVD:", svd_train.get_matrix(positions=[0]).shape)
assert svd_train.get_matrix(positions=[0]).shape[1] == SVD_COMPONENTS

In [ ]:
if INCLUDE_LDA:
    started = perf_counter()
    lda_operator = LDA(
        n_components=LDA_TOPICS, learning_method="batch",
        max_iter=LDA_ITERATIONS, random_state=RANDOM_STATE,
    )
    lda_train = project.translate(lda_operator, counts_train)["output"]
    timings["lda_fit"] = perf_counter() - started
    print("LDA:", lda_train.get_matrix(positions=[0]).shape)
    assert lda_train.get_matrix(positions=[0]).shape[1] == LDA_TOPICS
else:
    lda_operator = None
    lda_train = None
    print("Skipped LDA by configuration.")

In [ ]:
started = perf_counter()
augment_operator = FeatureMatrix(
    data_columns=True, metadata_columns=["log_words"], matrix_type="dense",
)
svd_plus_length_train = project.translate(
    augment_operator, svd_train, batch_size=QUERY_BATCH_SIZE,
)["output"]
timings["feature_matrix_fit"] = perf_counter() - started
assert svd_plus_length_train.get_matrix(positions=[0]).shape[1] == SVD_COMPONENTS + 1

representations = {
    "counts": counts_train,
    "tfidf": tfidf_train,
    "svd": svd_train,
    "svd_plus_length": svd_plus_length_train,
}
if INCLUDE_LDA:
    representations["lda"] = lda_train

display(pd.DataFrame([
    {"family": name, "artifact": rep.artifact_id,
     "rows": rep.n_rows, "features": len(rep.get_data_columns()),
     "storage": rep.artifact_type.value}
    for name, rep in representations.items()
]))
assert all(rep.n_rows == development.n_rows for rep in representations.values())

## 3. One cumulative classifier search, two targeted rounds

**Round 0 — representation comparison:** Compare L2-regularized multinomial
logistic regression with L-BFGS across all available TeAL matrix representations.
For each representation, search `C = [0.1, 1.0]`.

**Round 1 — regularization comparison:** Append new TF-IDF candidates using
SAGA, exploring L1 and 50/50 Elastic Net across the same values of `C`.
SAGA supports sparsity-inducing penalties but may converge extremely slowly on
unnormalized word-count matrices, as we found in diagnostic tests. We therefore
do **not** apply SAGA to raw counts.

The **same ClassifierSearch object** accumulates both rounds, retaining earlier
candidate IDs and scores without refitting them. Both rounds inherit identical
development labels, cross-validation folds and macro-F1 scoring. Results are
ranked together; finalization refits the selected model on the complete
development partition **before** any test labels are inspected.

We use `n_jobs=1` to avoid CPU oversubscription during diagnostics. scikit-learn
1.9's nested ProgressBar callbacks report search progress by fold/candidate;
SAGA does not expose within-fit iteration callbacks. Expected budget with five
representations: **10 + 4 = 14 candidates, 42 cross-validation fits**.
As before, convergence warnings should be examined; they are not evidence of
a meaningful inferiority of one penalty. All representation training happens
on development texts only.


In [ ]:
# Run a common L2 baseline over every representation first.
# This avoids the expensive SAGA + unnormalized raw-counts combination.
baseline_grid = {
    "C": [0.1, 1.0],
    "solver": ["lbfgs"],
    "max_iter": [500],
    "tol": [1e-3],
}
search = ClassifierSearch(
    representations=list(representations.values()),
    models={"logistic": baseline_grid},
    cv=CV_FOLDS, scoring="f1_macro",
    strategy="grid", n_jobs=1,
    random_state=RANDOM_STATE, batch_size=QUERY_BATCH_SIZE,
)
plan = search.plan(development, target="label")
print("Baseline candidates:", len(plan))
print("Baseline planned CV fits:", int(plan["planned_fits"].sum()))
print("A common L2 / L-BFGS baseline will run on all representations.")


In [ ]:
# Search round 0: fit L2 logistic regression on all feature representations.
started = perf_counter()
search.fit(development, target="label", progress=True)
timings["baseline_classifier_search"] = perf_counter() - started
baseline_scores = search.results().copy()
print("Baseline complete:", len(baseline_scores), "candidates")
display(baseline_scores[["candidate", "model", "params", "mean_score", "status"]])

# Search round 1: only TF-IDF, with sparsity-inducing regularization.
# No baseline models are retrained; all candidates share the same development
# labels, folds, scoring rule and final selection table.
sparse_penalty_grid = {
    "C": [0.1, 1.0],
    "solver": ["saga"],
    "l1_ratio": [0.5, 1.0],  # Elastic Net, L1
    "max_iter": [300],
    "tol": [1e-3],
}
if Version(sklearn.__version__) < Version("1.8"):
    sparse_penalty_grid["penalty"] = ["elasticnet"]

started = perf_counter()
search.fit_more(
    representations=[representations["tfidf"]],
    models={"logistic": sparse_penalty_grid},
    progress=True,
)
timings["regularization_classifier_search"] = perf_counter() - started
timings["classifier_search"] = (
    timings["baseline_classifier_search"]
    + timings["regularization_classifier_search"]
)

scores = search.results()
name_by_artifact = {rep.artifact_id: name for name, rep in representations.items()}
scores["representation_name"] = scores["representation_id"].map(name_by_artifact)

def regularization_label(row):
    ratio = row["params"].get("l1_ratio", 0.0)
    return "L2" if ratio == 0 else "L1" if ratio == 1 else f"Elastic Net ({ratio:g})"

scores["regularization"] = scores.apply(regularization_label, axis=1)
print("Combined candidates:", len(scores))
print("Combined attempted CV fits:", int(scores["executed_fits"].sum()))
display(scores.sort_values(
    ["mean_score", "std_score"], ascending=[False, True],
)[["candidate", "search_round", "representation_name", "model", "regularization",
    "params", "mean_score", "std_score", "fold_scores",
    "fit_seconds", "status", "error"]])
assert set(scores["status"]) == {"ok"}, scores.loc[scores["status"] != "ok", ["error"]]
assert (scores["search_round"].value_counts().sort_index().to_dict()
        == {0: len(representations) * 2, 1: 4})
display(scores.groupby(
    ["search_round", "representation_name", "regularization"], as_index=False,
)["mean_score"].agg(["count", "mean", "max"]).round(3))


## 4. Finalize on all development observations—before loading the test data

This is the decisive methodological boundary. `finalize(candidate=...)` creates a fresh estimator and fits it to all development labels, including the CV validation folds. Do not look at test classification results to revise this selection.

In [ ]:
best = scores.loc[scores["status"] == "ok"].sort_values(
    ["mean_score", "std_score"], ascending=[False, True],
).iloc[0]
winner = int(best["candidate"])
started = perf_counter()
chosen = search.finalize(candidate=winner)
timings["final_refit"] = perf_counter() - started
chosen_name = name_by_artifact[chosen.source.artifact_id]
print("Chosen representation:", chosen_name)
print("Selected model:", chosen.provenance["model"])
print("Selected regularization:", regularization_label(best))
print("Selected search round:", int(best["search_round"]))
print("Selected config:", chosen.provenance["params"])
print("Development rows:", chosen.provenance["development_rows"])
assert chosen.provenance["development_rows"] == len(train_texts)
assert chosen.predictor.training_provenance == chosen.provenance

### What regularization did to the selected model

L1 and Elastic Net often set some coefficients *exactly* to zero, whereas L2 typically retains small nonzero coefficients. Count nonzero entries only for the **final selected estimator**, recognizing that this depends on the chosen representation, regularization strength, convergence and feature scaling.

For multi-class linear models, coefficients usually have one row per category. Sparsity is therefore computed across the full coefficient matrix; zero coefficients are not a scientifically validated claim that those words are irrelevant.

In [ ]:
fitted = chosen.predictor.model
if hasattr(fitted, "coef_"):
    coefficients = np.asarray(fitted.coef_)
    nonzero = int(np.count_nonzero(coefficients))
    total = coefficients.size
    print("Estimator:", type(fitted).__name__)
    print("Coefficient array:", coefficients.shape)
    print("Exactly zero:", total - nonzero, "of", total)
    print("Coefficient sparsity:", f"{1 - nonzero / total:.1%}")
else:
    print("Selected estimator has no directly accessible coefficient array.")


## 5. Only now load the **independent** test partition

We now fetch the 20 Newsgroups `test` subset, which none of the learned representations has seen. The new TeAL source artifact includes **text and length metadata only**, never test category labels. Those labels remain in a separate DataFrame and are not used for any translation or refitting.

Each training-fitted translator is reapplied to the test artifacts. It should not refit or change vocabulary, feature names, SVD dimensions, or topic axes.

In [ ]:
started = perf_counter()
test_news = fetch_20newsgroups(
    subset="test", categories=None, shuffle=True, random_state=RANDOM_STATE,
    remove=("headers", "footers", "quotes"),
)
assert list(test_news.target_names) == names
test_texts = list(test_news.data)
test_gold = np.asarray(test_news.target, dtype=int)
if MAX_TEST is not None:
    test_texts = test_texts[:MAX_TEST]
    test_gold = test_gold[:MAX_TEST]
audit_truth = pd.DataFrame({
    "record_id": np.arange(len(test_texts), dtype=int),
    "gold": test_gold,
})
# NOTE: Test labels are deliberately absent from the input artifact.
audit_df = make_input_frame(test_texts)
assert "label" not in audit_df.columns
audit_csv = WORK / "audit_unlabeled.csv"
audit_df.to_csv(audit_csv, index=False)
audit_source = project.read_csv(
    audit_csv, text_fields="text",
    metadata_fields=["record_id", "log_words"],
    batch_size=QUERY_BATCH_SIZE,
)
assert audit_source.n_rows == len(audit_df)
timings["load_test"] = perf_counter() - started
print("Completely held-out documents:", audit_source.n_rows)

In [ ]:
started = perf_counter()
counts_test = project.translate(
    vectorizer, audit_source, batch_size=QUERY_BATCH_SIZE,
)["output"]
tfidf_test = project.translate(
    tfidf_operator, counts_test, batch_size=QUERY_BATCH_SIZE,
)["output"]
svd_test = project.translate(
    svd_operator, tfidf_test, batch_size=QUERY_BATCH_SIZE,
)["output"]
svd_plus_length_test = project.translate(
    augment_operator, svd_test, batch_size=QUERY_BATCH_SIZE,
)["output"]
test_representations = {
    "counts": counts_test,
    "tfidf": tfidf_test,
    "svd": svd_test,
    "svd_plus_length": svd_plus_length_test,
}
if INCLUDE_LDA:
    lda_test = project.translate(
        lda_operator, counts_test, batch_size=QUERY_BATCH_SIZE,
    )["output"]
    test_representations["lda"] = lda_test

timings["audit_representation_transform"] = perf_counter() - started
for name, train_rep in representations.items():
    test_rep = test_representations[name]
    assert train_rep.get_data_columns() == test_rep.get_data_columns(), name
    print(name, test_rep.n_rows, "rows,", len(test_rep.get_data_columns()), "features")

## 6. Predict on audit texts and evaluate by **keys**, not by array position

The fitted classifier can be applied to a different source with a compatible feature axis. The gold labels are joined only *after* predictions exist. We use the stable `record_id` metadata to check the join, while the TeAL primary key aligns source rows and prediction rows.

In [ ]:
started = perf_counter()
chosen_test_representation = test_representations[chosen_name]
audit_predictions = project.translate(
    chosen.predictor, chosen_test_representation,
    batch_size=QUERY_BATCH_SIZE,
)["output"]
timings["audit_prediction"] = perf_counter() - started

keys = list(audit_predictions.primary_key)
prediction_frame = audit_predictions.query(
    key_columns=True, data_columns=["prediction"], form="table",
)
audit_ids = audit_source.query(
    key_columns=True, data_columns=False,
    metadata_columns=["record_id"], metadata_mode="full", form="table",
)
matched = audit_ids.merge(prediction_frame, on=keys, validate="one_to_one")
scored = matched.merge(audit_truth, on="record_id", validate="one_to_one")
assert len(scored) == audit_source.n_rows
assert scored["prediction"].notna().all()
scored["actual_topic"] = scored["gold"].map(lambda n: names[int(n)])
scored["predicted_topic"] = scored["prediction"].map(lambda n: names[int(n)])

print("Independent test accuracy:", round(accuracy_score(scored["gold"], scored["prediction"]), 4))
print("Independent test macro F1:", round(f1_score(scored["gold"], scored["prediction"], average="macro"), 4))
display(scored[["record_id", "actual_topic", "predicted_topic"]].head(25))
print(classification_report(
    scored["gold"], scored["prediction"], target_names=names, zero_division=0,
))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    scored["gold"], scored["prediction"],
    display_labels=names, xticks_rotation=90,
    normalize="true", ax=ax, cmap="Blues",
)
ax.set_title("Full 20 Newsgroups: independent held-out topic confusion")
plt.tight_layout()
plt.show()
errors = scored.loc[scored["gold"] != scored["prediction"]]
display(errors[["record_id", "actual_topic", "predicted_topic"]].head(30))

## 7. Reopen the frozen fitted predictor and verify identical audit predictions

This exercises the last durability boundary. The operator's selection provenance should still identify the development labels, representation, hyperparameter configuration and development key fingerprint. Replaying the operator on the same held-out representation must reproduce the prediction vector.

In [ ]:
audit_rep_id = chosen_test_representation.artifact_id
op_id = project.catalog.get_operation(
    audit_predictions.descriptor["operation_id"]
)["operator_id"]
stored = prediction_frame.copy()
project.close()

reopened = Project.open(PROJECT_PATH)
loaded = reopened.get_operator(op_id)
assert loaded.training_provenance == chosen.provenance
replayed = reopened.translate(
    loaded, reopened.get_artifact(audit_rep_id),
    batch_size=QUERY_BATCH_SIZE,
)["output"]
fresh = replayed.query(
    key_columns=True, data_columns=["prediction"], form="table",
)
merged = stored.merge(fresh, on=keys, suffixes=("_original", "_replayed"),
                      validate="one_to_one")
np.testing.assert_array_equal(
    merged["prediction_original"], merged["prediction_replayed"]
)
print("Frozen predictions reproduced on", len(merged), "test documents.")
reopened.close()

## 8. Resource report and interpretation

The full run can take substantial time, particularly LDA and L1/Elastic Net cross-validation across multiple feature spaces. This is expected, and is why the notebook is **local/manual** rather than a required GitHub CI gate.

Interpret the confusion matrix, regularization choice, coefficient sparsity and per-class precision/recall—not just a single aggregate number. Distinguish:
- feature representation learned from *development text*;
- hyperparameter selection using *development labels*;
- final model refit on *all development labels*;
- independent evaluation using *previously unused test texts and labels*.

There is still a finer nested-CV issue: the development representation is held fixed across folds. The independent test set, transformed using the frozen operators, provides a cleaner check of generalization to new documents.

In [ ]:
display(pd.DataFrame([
    {"stage": stage, "seconds": round(elapsed, 2)}
    for stage, elapsed in timings.items()
]).sort_values("seconds", ascending=False))
print("Data and TeAL artifacts are in:", WORK)
print("All notebook assertions passed. Inspect the diagnostics before drawing conclusions.")